# Evaluasi model DiT

Notebook ini dapat diimpor langsung ke Kaggle. Notebook hanya membaca hasil eksperimen, tanpa training ulang. Gunakan hasil run final yang menghasilkan submission pilihanmu.


## 1. Tentukan folder hasil
Jika memakai sesi training yang sama, gunakan path bawaan. Jika hasil berasal dari sesi sebelumnya, tambahkan hasil notebook tersebut sebagai Input Kaggle, lalu ubah `OUTPUT_DIR` menjadi folder yang berisi file evaluasi.


In [ ]:
from pathlib import Path
OUTPUT_DIR = Path("/kaggle/working/dit_pretrained")
REPORT_DIR = Path("/kaggle/working/reports")
REPORTED_PUBLIC_SCORE = None  # Isi setelah submission run terbaru dinilai Kaggle.


In [ ]:
import json
import shutil
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
REPORT_DIR.mkdir(parents=True, exist_ok=True)
required = ["metrics.json", "history.csv", "classification_report.csv", "confusion_matrix.csv", "validation_predictions.csv"]
missing = [name for name in required if not (OUTPUT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"File hasil belum lengkap di {OUTPUT_DIR}: {missing}. Periksa lokasi hasil run Kaggle.")


## 2. Hasil validation
Public score dan validation accuracy berasal dari data penilaian berbeda. Tabel per kelas memperlihatkan precision, recall, dan F1-score dari run yang dibaca.


In [ ]:
metrics = json.loads((OUTPUT_DIR / "metrics.json").read_text())
print("Validation accuracy:", metrics["accuracy"])
print("Validation macro-F1:", metrics["macro_f1"])
print("Public score run terbaru:", REPORTED_PUBLIC_SCORE if REPORTED_PUBLIC_SCORE is not None else "belum dilaporkan")
report = pd.read_csv(OUTPUT_DIR / "classification_report.csv", index_col=0)
display(report)


## 3. Perkembangan training
Bandingkan loss train dan validation. Validation loss yang naik saat train loss turun dapat menjadi tanda model terlalu menyesuaikan diri dengan data train.


In [ ]:
history = pd.read_csv(OUTPUT_DIR / "history.csv")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["epoch"], history["train_loss"], label="Train loss")
axes[0].plot(history["epoch"], history["val_loss"], label="Validation loss")
axes[0].set_ylabel("Loss")
axes[1].plot(history["epoch"], history["accuracy"], label="Accuracy")
axes[1].plot(history["epoch"], history["macro_f1"], label="Macro-F1")
axes[1].set_ylabel("Skor")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.legend()
fig.tight_layout()
fig.savefig(REPORT_DIR / "training_curve.png", dpi=150)
plt.show()
plt.close(fig)


## 4. Confusion matrix
Baris menunjukkan kategori sebenarnya dan kolom menunjukkan prediksi. Nilai di luar diagonal merupakan prediksi yang salah.


In [ ]:
matrix = pd.read_csv(OUTPUT_DIR / "confusion_matrix.csv", index_col=0)
fig, ax = plt.subplots(figsize=(12, 10))
image = ax.imshow(matrix.to_numpy(), cmap="Blues")
ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=90)
ax.set_yticks(range(len(matrix.index)), matrix.index)
ax.set_xlabel("Prediksi")
ax.set_ylabel("Kategori sebenarnya")
fig.colorbar(image, ax=ax)
fig.tight_layout()
fig.savefig(REPORT_DIR / "confusion_matrix.png", dpi=150)
plt.show()
plt.close(fig)


## 5. Periksa kesalahan per kelas
Tabel pertama diurutkan berdasarkan recall. Tabel kedua memperlihatkan contoh prediksi salah dengan confidence tinggi. Confidence adalah probabilitas terbesar yang diberikan model, bukan jaminan prediksi benar.


In [ ]:
class_report = report.loc[matrix.index, ["precision", "recall", "f1-score", "support"]]
display(class_report.sort_values("recall"))
predictions = pd.read_csv(OUTPUT_DIR / "validation_predictions.csv", dtype={"id": str})
errors = predictions.loc[predictions["label"].ne(predictions["predicted_label"])]
display(errors.sort_values("confidence", ascending=False).head(20))
errors.to_csv(REPORT_DIR / "validation_errors.csv", index=False)


## 6. Simpan laporan
File berikut dibuat dari hasil yang dibaca, bukan angka perkiraan. Unduh grafik dan tabel dari folder reports, lalu salin ke folder reports pada repository GitHub.


In [ ]:
for name in ["metrics.json", "history.csv", "classification_report.csv", "confusion_matrix.csv", "run_config.json"]:
    source = OUTPUT_DIR / name
    if source.is_file():
        shutil.copy2(source, REPORT_DIR / name)
summary = [
    "# Ringkasan evaluasi DiT", "",
    f"Validation accuracy: {metrics['accuracy']:.4f}.",
    f"Validation macro-F1: {metrics['macro_f1']:.4f}.",
    f"Public score run terbaru: {REPORTED_PUBLIC_SCORE if REPORTED_PUBLIC_SCORE is not None else 'belum dilaporkan'}.",
    f"Jumlah sampel validation: {len(predictions):,}.",
    f"Jumlah prediksi salah: {len(errors):,}.",
    "",
    "Hasil per kategori tersedia pada classification_report.csv.",
    "Grafik training dan confusion matrix dibuat dari berkas run yang dibaca.",
]
(REPORT_DIR / "validation_summary.md").write_text("\n".join(summary) + "\n", encoding="utf-8")
print("Grafik dan tabel disimpan di:", REPORT_DIR)
